# Baseline Models - Auto-Urgency Tagging (LEN-05)

**Assignee:** Cokorda  
**Sprint:** 2 (18 Mei - 24 Mei)  
**PBI:** LEN-05 — Pelatihan Model Baseline Klasik untuk prediksi tingkat urgensi keluhan nasabah (High, Medium, Low).

Notebook ini mengintegrasikan pipeline rekayasa fitur yang telah dibangun oleh **Afif (LEN-04)** pada berkas `src/features.py`, kemudian melatih dan mengevaluasi dua algoritma ML klasik:
1. **Logistic Regression** — cepat, interpretatif, sebagai acuan minimum
2. **Random Forest** — ensemble berbasis pohon, lebih robust terhadap noise

Kedua model dikonfigurasi dengan `class_weight='balanced'` untuk menangani ketidakseimbangan distribusi kelas.

---
## Cell 1 — Setup Lingkungan & Import Library

In [ ]:
import sys
import os

# Deteksi apakah berjalan di Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Jalur absolut menuju direktori root proyek Anda di Drive
    PROJECT_ROOT = '/content/drive/MyDrive/lentera-analytics-hub/lentera-ml-research'
    if not os.path.exists(PROJECT_ROOT):
        PROJECT_ROOT = '/content/drive/MyDrive/lentera-ml-research'
        
    os.chdir(PROJECT_ROOT)
    print(f"Berjalan di Google Colab. Direktori aktif saat ini: {os.getcwd()}")
else:
    # Berjalan secara lokal (VS Code / Jupyter Notebook)
    # CWD biasanya berada di folder 'notebooks', jadi root berada satu tingkat di atasnya
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print(f"Berjalan secara lokal. Project root: {PROJECT_ROOT}")

# Tambahkan root proyek ke sys.path jika belum terdaftar
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# --- Library Umum ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Pipeline Rekayasa Fitur (LEN-04, oleh Afif) ---
from src.features import split_data, extract_tfidf

# --- Model ML Klasik (Scikit-learn) ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Styling untuk visualisasi
sns.set_theme(style='whitegrid')
print("Semua library berhasil diimport.")

---
## Cell 2 — Load & Validasi Dataset Bersih

In [ ]:
# Membaca dataset yang telah melalui proses pembersihan teks (output dari LEN-03)
# yang disimpan di direktori `data/processed/` oleh tim preprocessing.
DATA_PATH = '../data/processed/cleaned_complaints.csv'

df = pd.read_csv(DATA_PATH)
print(f"Dataset berhasil dimuat. Shape awal: {df.shape}")
print(f"Kolom tersedia: {df.columns.tolist()}")
print()

# Validasi: hapus baris dengan nilai kosong pada kolom kunci.
# Kolom `Cleaned_Narrative` adalah input teks dan `Urgency` adalah target label.
df = df.dropna(subset=['Cleaned_Narrative', 'Urgency'])
print(f"Shape setelah dropna: {df.shape}")
print()

# Cek distribusi kelas untuk memahami tingkat ketidakseimbangan dataset
print("Distribusi kelas Urgency:")
print(df['Urgency'].value_counts())

---
## Cell 3 — Visualisasi Distribusi Kelas

In [ ]:
# Memvisualisasikan distribusi kelas target untuk mengkonfirmasi adanya
# ketidakseimbangan kelas (class imbalance) yang menjadi dasar penggunaan
# parameter `class_weight='balanced'` pada model.
order = ['High', 'Medium', 'Low']
counts = df['Urgency'].value_counts().reindex(order)

plt.figure(figsize=(7, 4))
bars = sns.barplot(x=counts.index, y=counts.values, palette='viridis', hue=counts.index, legend=False)

# Tambahkan label angka di atas setiap bar untuk kemudahan pembacaan
for bar in bars.patches:
    bars.annotate(
        f'{int(bar.get_height()):,}',
        (bar.get_x() + bar.get_width() / 2., bar.get_height()),
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

plt.title('Distribusi Kelas Target: Urgency', fontsize=14, fontweight='bold')
plt.xlabel('Tingkat Urgensi')
plt.ylabel('Jumlah Data')
plt.tight_layout()
plt.show()

---
## Cell 4 — Data Splitting (Pipeline LEN-04)

In [ ]:
# Memanggil fungsi `split_data()` dari pipeline LEN-04 (src/features.py).
# Fungsi ini membagi dataset secara stratified dengan rasio:
#   - Train  : 70%
#   - Val    : 15%
#   - Test   : 15%
df_train, df_val, df_test = split_data(
    df,
    text_col='Cleaned_Narrative',
    target_col='Urgency'
)

# Ekstrak label target (y) dari masing-masing split DataFrame
y_train = df_train['Urgency']
y_val   = df_val['Urgency']
y_test  = df_test['Urgency']

print("\nDistribusi kelas pada setiap split:")
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f"  {name}: {y.value_counts().to_dict()}")

---
## Cell 5 — Ekstraksi Fitur TF-IDF (Pipeline LEN-04)

In [ ]:
# Memanggil fungsi `extract_tfidf()` dari pipeline LEN-04 (src/features.py).
# Fungsi ini melakukan fit vectorizer HANYA pada data training untuk menghindari
# data leakage, lalu mentransformasi data val dan test menggunakan vocabulari
# yang sama dari training set.
X_train_tfidf, X_val_tfidf, X_test_tfidf, tfidf_vectorizer = extract_tfidf(
    train_texts=df_train['Cleaned_Narrative'],
    val_texts=df_val['Cleaned_Narrative'],
    test_texts=df_test['Cleaned_Narrative']
)

print(f"Shape fitur TF-IDF:")
print(f"  X_train : {X_train_tfidf.shape}")
print(f"  X_val   : {X_val_tfidf.shape}")
print(f"  X_test  : {X_test_tfidf.shape}")
print(f"\nUkuran vocabulary TF-IDF: {len(tfidf_vectorizer.vocabulary_)} kata")

---
## Cell 6 — Model 1: Logistic Regression

In [ ]:
# --- PELATIHAN: Logistic Regression ---
# Logistic Regression dipilih sebagai baseline minimum yang cepat dan interpretatif.
# Parameter penting:
#   - class_weight='balanced'  : mengkompensasi ketidakseimbangan kelas secara otomatis
#                                dengan menaikkan bobot pada kelas minoritas.
#   - max_iter=1000            : iterasi lebih banyak agar model konvergen pada data besar.
#   - random_state=42          : reproducibility hasil.
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

print("Melatih Logistic Regression...")
lr_model.fit(X_train_tfidf, y_train)
print("Pelatihan selesai.")

---
## Cell 7 — Evaluasi Logistic Regression pada Test Set

In [ ]:
# Melakukan prediksi pada test set menggunakan model Logistic Regression
# yang telah dilatih. Test set bersifat held-out dan TIDAK pernah dilihat model
# selama pelatihan maupun pemilihan hyperparameter.
y_pred_lr = lr_model.predict(X_test_tfidf)

print("=" * 55)
print("  EVALUASI MODEL 1: Logistic Regression (Test Set)")
print("=" * 55)

# Classification Report menampilkan Precision, Recall, F1-Score per kelas
# serta rata-rata macro dan weighted — metrik utama untuk dataset imbalanced.
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=['High', 'Medium', 'Low']
))

---
## Cell 8 — Confusion Matrix Logistic Regression

In [ ]:
# Visualisasi Confusion Matrix untuk melihat pola kesalahan prediksi
# antar kelas secara lebih intuitif.
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_lr,
    display_labels=['High', 'Medium', 'Low'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 9 — Model 2: Random Forest

In [ ]:
# --- PELATIHAN: Random Forest ---
# Random Forest adalah model ensemble yang lebih kompleks dan umumnya lebih
# robust terhadap noise dibandingkan Logistic Regression.
# Parameter penting:
#   - n_estimators=200         : jumlah pohon keputusan dalam ensemble.
#   - class_weight='balanced'  : penanganan class imbalance (wajib, sama seperti LR).
#   - n_jobs=-1                : memanfaatkan seluruh core CPU untuk mempercepat pelatihan.
#   - random_state=42          : reproducibility hasil.
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

print("Melatih Random Forest (200 trees)... Ini mungkin memerlukan beberapa menit.")
rf_model.fit(X_train_tfidf, y_train)
print("Pelatihan selesai.")

---
## Cell 10 — Evaluasi Random Forest pada Test Set

In [ ]:
# Melakukan prediksi pada test set menggunakan model Random Forest
y_pred_rf = rf_model.predict(X_test_tfidf)

print("=" * 55)
print("  EVALUASI MODEL 2: Random Forest (Test Set)")
print("=" * 55)

# Classification Report: Precision, Recall, F1-Score per kelas
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=['High', 'Medium', 'Low']
))

---
## Cell 11 — Confusion Matrix Random Forest

In [ ]:
# Visualisasi Confusion Matrix untuk Random Forest
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_rf,
    display_labels=['High', 'Medium', 'Low'],
    cmap='Greens',
    ax=ax
)
ax.set_title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 12 — Perbandingan Performa Kedua Model (Ringkasan)

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Membuat tabel ringkasan perbandingan metrik utama kedua model
# untuk memudahkan pengambilan keputusan model mana yang akan
# digunakan lebih lanjut (handoff ke tim evaluasi - LEN-09).
results = {
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ],
    'F1-Score Macro': [
        f1_score(y_test, y_pred_lr, average='macro'),
        f1_score(y_test, y_pred_rf, average='macro')
    ],
    'F1-Score Weighted': [
        f1_score(y_test, y_pred_lr, average='weighted'),
        f1_score(y_test, y_pred_rf, average='weighted')
    ]
}

df_results = pd.DataFrame(results)
df_results[['Accuracy', 'F1-Score Macro', 'F1-Score Weighted']] = \
    df_results[['Accuracy', 'F1-Score Macro', 'F1-Score Weighted']].applymap(lambda x: f'{x:.4f}')

print("\n=" * 55)
print("  RINGKASAN PERBANDINGAN PERFORMA MODEL BASELINE")
print("=" * 55)
print(df_results.to_string(index=False))
print()
print("Catatan: F1-Score Macro diprioritaskan sebagai metrik utama")
print("         karena dataset memiliki ketidakseimbangan kelas.")

---
## Cell 13 — Visualisasi Perbandingan F1-Score per Kelas

In [ ]:
from sklearn.metrics import classification_report

# Mengekstrak F1-Score per kelas dari kedua model untuk visualisasi
# perbandingan side-by-side yang lebih mudah dibaca.
classes = ['High', 'Medium', 'Low']

report_lr = classification_report(y_test, y_pred_lr, target_names=classes, output_dict=True)
report_rf = classification_report(y_test, y_pred_rf, target_names=classes, output_dict=True)

f1_lr = [report_lr[c]['f1-score'] for c in classes]
f1_rf = [report_rf[c]['f1-score'] for c in classes]

x = np.arange(len(classes))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, f1_lr, width, label='Logistic Regression', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, f1_rf, width, label='Random Forest', color='seagreen', alpha=0.85)

# Tambahkan nilai di atas setiap bar
for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', fontsize=10)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', fontsize=10)

ax.set_xlabel('Kelas Urgency')
ax.set_ylabel('F1-Score')
ax.set_title('Perbandingan F1-Score per Kelas\nLogistic Regression vs Random Forest', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.show()